In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())

import matplotlib.pyplot as plt
from dataset import MOViC_Dataset
from torchvision import transforms
from torchvision.transforms import (
    RandomHorizontalFlip,
    RandomVerticalFlip,
    RandomRotation,
    ColorJitter,
    ToTensor,
    Resize
)
import numpy as np

import imageio
from io import BytesIO
from IPython.display import display, Image as IPImage
from PIL import Image

True


In [33]:
DATASET_PATH = '/home/nfs/inf6/data/datasets/MOVi/movi_c'

In [34]:
EXP_DIR = 'experiments_released'
models = {
    "Pred_Obj_AR": f"{EXP_DIR}/30-09-2025_11-09_TransformerPredictorObjAutoRegressiveMask0.01_MLP2048",
    "Pred_Obj_TF": f"{EXP_DIR}/30-09-2025_14-11_TransformerPredictorObjTeacherForceeMask0.01_MLP2048",
    "Pred_RGB_AR": f"{EXP_DIR}/30-09-2025_06-39_TransformerPredictorRGBAutoRegressive",
    "Pred_RGB_TF": f"{EXP_DIR}/30-09-2025_14-14_TransformerPredictorRGBTeacherForce"
}

In [36]:
from typing import Iterable, List, Optional, Tuple, Union

class _Runner:
    def __init__(self, model, writer, dir_imgs: str):
        self.model = model
        self.writer = writer
        self.dir_imgs = dir_imgs

    @torch.no_grad()
    def _log_predictions_rgb(self, batch, iter_):
        # ====== ТВОЙ КОД ======
        context = batch[0, :5].unsqueeze(0)   # (1, 5, C, H, W)
        future  = batch[0, 5:].unsqueeze(0)   # (1, 15, C, H, W)
        B, _, C, H, W = context.shape

        preds = []
        cur_context = context.clone()
        with torch.no_grad():
            for t in range(15):
                out = self.model(cur_context)             # [1, T=5, C, H, W]
                next_frame = out[:, -1]                   # [1,C,H,W]
                preds.append(next_frame.unsqueeze(1))     # [1,1,C,H,W]
                cur_context = torch.cat([cur_context[:, 1:], next_frame.unsqueeze(1)], dim=1)

        preds = torch.cat(preds, dim=1)  # (1, 15, C, H, W)

        ctx_seq   = context[0]      # (5, C, H, W)
        fut_seq   = future[0]       # (15, C, H, W)
        pred_seq  = preds[0]        # (15, C, H, W)

        draw_rollout_grid(ctx_seq, fut_seq, pred_seq, self.writer, self.dir_imgs, iter_, mode='val')

        # ----- PRED GIF -----
        scale = 4
        seq_pred = torch.cat([ctx_seq, pred_seq], dim=0) \
            .clamp(0,1).mul(255).byte().cpu().permute(0,2,3,1).numpy()
        images = []
        for f in seq_pred:
            img = Image.fromarray(f)
            if scale != 1.0:
                w, h = img.size
                img = img.resize((int(w * scale), int(h * scale)), Image.NEAREST)
            images.append(img)
        imageio.mimsave(os.path.join(self.dir_imgs, f"rollout_val_{iter_:06d}.gif"), images, fps=5)

        # ----- GT GIF (новое) -----
        seq_gt = torch.cat([ctx_seq, fut_seq], dim=0) \
            .clamp(0,1).mul(255).byte().cpu().permute(0,2,3,1).numpy()
        images_gt = []
        for f in seq_gt:
            img = Image.fromarray(f)
            if scale != 1.0:
                w, h = img.size
                img = img.resize((int(w * scale), int(h * scale)), Image.NEAREST)
            images_gt.append(img)
        imageio.mimsave(os.path.join(self.dir_imgs, f"rollout_gt_{iter_:06d}.gif"), images_gt, fps=5)

    @torch.no_grad()
    def _log_predictions_obj(self, batch, iter_):
        # ====== ТВОЙ КОД ======
        frames, masks = batch
        context = (frames[0:1, :5], masks[0:1, :5])   # ([1,5,C,H,W], [1,5,H,W])
        future_f = frames[0:1, 5:20]                  # [1,15,C,H,W]
        future_m = masks[0:1,  5:20]                  # [1,15,H,W]

        preds_f, preds_m = [], []
        cur_context = (context[0].clone(), context[1].clone())

        self.model.eval()
        with torch.no_grad():
            for _ in range(future_f.size(1)):
                out_f, out_m, _ = self.model(cur_context)       # ([1,5,C,H,W], [1,5,H,W])
                next_f = out_f[:, -1]                        # [1,C,H,W]
                next_m = out_m[:, -1]                        # [1,H,W]
                preds_f.append(next_f.unsqueeze(1))          # [1,1,C,H,W]
                preds_m.append(next_m.unsqueeze(1))          # [1,1,H,W]
                cur_context = (
                    torch.cat([cur_context[0][:, 1:], next_f.unsqueeze(1)], dim=1),
                    torch.cat([cur_context[1][:, 1:], next_m.unsqueeze(1)], dim=1),
                )

        preds_f = torch.cat(preds_f, dim=1)[0]  # [15,C,H,W]
        preds_m = torch.cat(preds_m, dim=1)[0]  # [15,H,W]
        ctx_f, fut_f = context[0][0], future_f[0]  # [5,C,H,W], [15,C,H,W]
        ctx_m, fut_m = context[1][0], future_m[0]  # [5,H,W],   [15,H,W]

        # кадры
        draw_rollout_grid(ctx_f, fut_f, preds_f, self.writer, self.dir_imgs, iter_, mode='val')

        # маски (цветные)
        ctx_m_rgb  = masks_to_rgb_tensor(ctx_m)   # [5,3,H,W]
        fut_m_rgb  = masks_to_rgb_tensor(fut_m)   # [15,3,H,W]
        preds_m_rgb= masks_to_rgb_tensor(preds_m) # [15,3,H,W]
        draw_rollout_grid(ctx_m_rgb, fut_m_rgb, preds_m_rgb, self.writer, self.dir_imgs, iter_, mode='val_masks')

        # ----- PRED frames GIF -----
        scale = 4
        seq_pred_f = (torch.cat([ctx_f, preds_f], dim=0).clamp(0,1)
                      .mul(255).byte().cpu().permute(0,2,3,1).numpy())
        images = []
        for f in seq_pred_f:
            img = Image.fromarray(f)
            if scale != 1.0:
                w, h = img.size
                img = img.resize((int(w*scale), int(h*scale)), Image.NEAREST)
            images.append(img)
        imageio.mimsave(os.path.join(self.dir_imgs, f"rollout_val_{iter_:06d}.gif"), images, fps=5)

        # ----- PRED masks GIF -----
        seq_pred_m = (torch.cat([ctx_m_rgb, preds_m_rgb], dim=0)
                      .clamp(0,1).mul(255).byte().cpu().permute(0,2,3,1).numpy())
        images_m = []
        for f in seq_pred_m:
            img = Image.fromarray(f)
            if scale != 1.0:
                w, h = img.size
                img = img.resize((int(w*scale), int(h*scale)), Image.NEAREST)
            images_m.append(img)
        imageio.mimsave(os.path.join(self.dir_imgs, f"rollout_val_masks_{iter_:06d}.gif"), images_m, fps=5)

        # ----- GT frames GIF (новое) -----
        seq_gt_f = (torch.cat([ctx_f, fut_f], dim=0).clamp(0,1)
                    .mul(255).byte().cpu().permute(0,2,3,1).numpy())
        images_gt_f = []
        for f in seq_gt_f:
            img = Image.fromarray(f)
            if scale != 1.0:
                w, h = img.size
                img = img.resize((int(w*scale), int(h*scale)), Image.NEAREST)
            images_gt_f.append(img)
        imageio.mimsave(os.path.join(self.dir_imgs, f"rollout_gt_{iter_:06d}.gif"), images_gt_f, fps=5)

        # ----- GT masks GIF (новое) -----
        seq_gt_m = (torch.cat([ctx_m_rgb, fut_m_rgb], dim=0)
                    .clamp(0,1).mul(255).byte().cpu().permute(0,2,3,1).numpy())
        images_gt_m = []
        for f in seq_gt_m:
            img = Image.fromarray(f)
            if scale != 1.0:
                w, h = img.size
                img = img.resize((int(w*scale), int(h*scale)), Image.NEAREST)
            images_gt_m.append(img)
        imageio.mimsave(os.path.join(self.dir_imgs, f"rollout_gt_masks_{iter_:06d}.gif"), images_gt_m, fps=5)

@torch.no_grad()
def log_rollouts_by_indices(
    model: torch.nn.Module,
    TYPE: str,                      # 'RGB' | 'Obj'
    MODE: str,                      # 'AR' | 'TF' (только для имени папки)
    validation_loader,              # любой PyTorch DataLoader
    draw_rollout_grid,              # твоя функция
    masks_to_rgb_tensor=None,       # нужна для Obj
    indices: List[int] = [0],       # индексы элементов в validation_loader.dataset
    save_root: str = "results",
    writer=None,
    device: Optional[torch.device] = None,
):
    """
    Берём элементы напрямую из validation_loader.dataset по indices
    и логируем в results/Pred_{TYPE}_{MODE}/validation_{k}/
    """
    assert TYPE in ("RGB", "Obj")
    dataset = validation_loader.dataset

    if device is not None:
        model.to(device)
    else:
        device = next(model.parameters()).device
    model.eval()

    base_dir = os.path.join(save_root, f"Pred_{TYPE}_{MODE}")
    os.makedirs(base_dir, exist_ok=True)

    for k, idx in enumerate(indices):
        out_dir = os.path.join(base_dir, f"validation_{k:06d}")
        os.makedirs(out_dir, exist_ok=True)

        # --- достаём сэмпл из dataset ---
        sample = dataset[idx]

        # разные возможные форматы датасета:
        if TYPE == "RGB":
            if isinstance(sample, dict) and "frames" in sample:
                frames = sample["frames"]
            elif isinstance(sample, (tuple, list)) and torch.is_tensor(sample[0]):
                frames = sample[0]
            else:
                frames = sample  # предполагаем Tensor[T,C,H,W]

            assert torch.is_tensor(frames) and frames.ndim == 4, \
                f"Ожидаю Tensor[T,C,H,W] для RGB, получил {type(frames)} shape={getattr(frames,'shape',None)}"

            # мини-батч [1,T,C,H,W], как ожидает твой код
            batch = frames.unsqueeze(0).to(device, non_blocking=True)

            runner = _Runner(model=model, writer=writer, dir_imgs=out_dir)
            runner._log_predictions_rgb(batch, iter_=0)

        else:  # Obj
            # формы: (frames[T,C,H,W], masks[T,H,W]) или dict
            if isinstance(sample, dict) and "frames" in sample and "masks" in sample:
                frames = sample["frames"]
                masks = sample["masks"]
            elif isinstance(sample, (tuple, list)) and len(sample) >= 2:
                frames, masks = sample[0], sample[1]
            else:
                raise ValueError("Для Obj ожидаю (frames, masks) или dict с ключами 'frames','masks'")

            assert torch.is_tensor(frames) and frames.ndim == 4, "frames должен быть [T,C,H,W]"
            assert torch.is_tensor(masks) and masks.ndim == 3, "masks должен быть [T,H,W]"

            # мини-батч: ([1,T,C,H,W], [1,T,H,W])
            f_sel = frames.unsqueeze(0).to(device, non_blocking=True)
            m_sel = masks.unsqueeze(0).to(device, non_blocking=True)
            if m_sel.dtype != torch.long:
                m_sel = m_sel.long()

            runner = _Runner(model=model, writer=writer, dir_imgs=out_dir)
            runner._log_predictions_obj((f_sel, m_sel), iter_=0)


In [37]:
from models.masked_object_transformer import MaskedObjectTransformer
from models.encoder import ViTPatchEncoder
from models.decoder import ViTPatchDecoder
from models.vpt import VideoFrameTransformer, VideoObjectTransformer
from save_load import load_model



TYPE = 'RGB' #Obj or RGB
MODE = 'TF' #AR or TF
EMB_DIM = 512 if TYPE=='Obj' else 256
PATCH_SIZE = 8
predictor, encoder, decoder = None, None, None
if TYPE=='Obj':
    autoencoder = MaskedObjectTransformer(obj_num=11, embed_dim=EMB_DIM,
                                          img_size=64, enc_depth=3,
                                          nhead=8, mlp_ratio=4.0)
    encoder, decoder = autoencoder.encoder, autoencoder.decoder
elif TYPE=='RGB':
    PATCH_SIZE = 8
    encoder = ViTPatchEncoder(patch_size=PATCH_SIZE,
                              embed_dim=EMB_DIM,
                              max_len=(64 // PATCH_SIZE) * (64 // PATCH_SIZE),
                              attn_dim=32,
                              num_heads=8,
                              mlp_size=1024,
                              num_tf_layers=12)

    decoder = ViTPatchDecoder(patch_size=PATCH_SIZE,
                              H = 64,
                              W = 64,
                              embed_dim=EMB_DIM,
                              max_len=(64 // PATCH_SIZE) * (64 // PATCH_SIZE),
                              attn_dim=32,
                              num_heads=8,
                              mlp_size=1024,
                              num_tf_layers=12)

if TYPE=='Obj':
    predictor = VideoObjectTransformer(
        encoder=encoder,
        decoder=decoder,
        embed_dim=EMB_DIM,
        attn_dim=EMB_DIM // 8, #emb // num_of_heads!!!!!!!!!!!!!! 52
        num_heads=8,
        mlp_size=4 * EMB_DIM,
        num_tf_layers_ar=12
    )
else:
    predictor =  VideoFrameTransformer(
        encoder=encoder,
        decoder=decoder,
        H=64, W=64,
        patch_size=PATCH_SIZE,
        embed_dim=EMB_DIM,
        attn_dim=EMB_DIM//8,
        num_heads=8,
        mlp_size=4*EMB_DIM,
        num_tf_layers_ar=12
    )

model_path = models[f'Pred_{TYPE}_{MODE}']+'/checkpoints'

predictor, _, _ = load_model(predictor, None, 'epoch_020_iter_24360', model_path)

In [38]:
from torch.utils.data import DataLoader

TARGET = 'objects' if TYPE=='Obj' else 'rgb'
B_SIZE = 8
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
val_dataset = MOViC_Dataset(root_dir=DATASET_PATH, split="validation", target=TARGET, img_size=(64, 64), target_frames=15, transform=val_transform)

val_loader = DataLoader(
        dataset=val_dataset, 
        batch_size=B_SIZE,
        shuffle=False,
        num_workers=4
)

In [41]:
from utils.visualize_video_sample import draw_rollout_grid
import os
log_rollouts_by_indices(
    model=predictor,
    TYPE=TYPE,
    MODE=MODE,
    validation_loader=val_loader,
    draw_rollout_grid=draw_rollout_grid,
    indices=list(range(1, 30, 2)),
    save_root="results",
    writer=None,
    device=torch.device('cuda:0'),
)